# Lab 2 — Turn It Into a Real Gamepad

In Lab 1 you proved your inputs work. Now for the magic trick: we'll make the computer believe a **real USB game controller** is plugged in — and *you* will be the one feeding it inputs.

**How can Python pretend to be a gamepad?** The operating system has a feature (called `uinput`) that lets a program create *virtual* devices. When we use it, every game on the machine sees a controller named **"Workshop Controller"**, exactly as if you'd plugged one in. The game never knows the difference — which means *any* input you can read in Python can drive *any* game. That idea is the heart of accessible gaming: the game doesn't need to change, the controller does.

The plan:

1. Paste in your verified setup from Lab 1.
2. Create the virtual gamepad.
3. Write the **game loop**: read your inputs, hand them to the gamepad, repeat forever.
4. Play SuperTuxKart with the controller you built.

## Step 1 — Bring over your verified setup

In Lab 1, your **Step 1 setup cell** (the one starting with `YOUR CONTROLLER SETUP`) recorded exactly which inputs you built and which pins they use — and you *verified* those pins with the live test. Paste that whole cell into the code cell below, replacing the placeholder comments. Then run it.

In [ ]:
# =====================================================
#  PASTE YOUR SETUP CELL FROM LAB 1 HERE
#  (it defines LEFT_INPUT, LEFT_PINS, RIGHT_INPUT, RIGHT_PINS)
# =====================================================



# If the next cell complains "NameError: LEFT_INPUT is not defined",
# it means this paste step was skipped!

## Step 2 — Wake up the hardware and create the gamepad

Same connect call as Lab 1, plus one new line that creates the virtual controller. (If it complains about `/dev/uinput`, open a terminal, run `sudo modprobe uinput`, and run this cell again.)

If your controller uses the tilt sensor, **hold it flat and still** while this runs — it zeroes itself during connect.

In [ ]:
import workshopHelpers as ws

ws.connect(LEFT_INPUT, LEFT_PINS, RIGHT_INPUT, RIGHT_PINS)
ws.make_gamepad()

## Step 3 — Build your game loop

Every controller in the world works the same way, thousands of times per second:

> **read the inputs → tell the game → repeat**

That's called the **game loop**, and you're about to write yours. Our virtual gamepad has the same parts as a store-bought one:

| Gamepad part | How you move it in code |
|---|---|
| Left stick | `ws.left_stick(x, y)` — x and y from -1.0 to +1.0 |
| Right stick | `ws.right_stick(x, y)` |
| Triggers | `ws.trigger("left", amount)` — amount from 0.0 to 1.0 |
| Buttons `A B X Y L R LS RS` | `ws.button("A", True_or_False)` |

### Your job

The loop cell below is pre-filled with an **example** build (joystick on the left, buttons on the right). Replace each half with the snippet matching **your** input on that side, from this menu — and use **your** pin numbers where marked:

**Joystick** (as your LEFT input — it drives the left stick):

```python
x, y = ws.read_joystick_percent("left")
ws.left_stick(x, y)
ws.button("LS", ws.read_button(36))     # <- YOUR SW pin
```

*(on the RIGHT: use `"right"`, `ws.right_stick`, and button `"RS"`)*

**Rotary encoder** (as your LEFT input — the knob acts like sliding the stick up/down, and its press is the `L` shoulder button):

```python
ws.left_stick(0, ws.read_dial_percent("left"))
ws.button("L", ws.read_button(19))      # <- YOUR SW pin
```

*(on the RIGHT: use `"right"`, `ws.right_stick`, and button `"R"`)*

**Two buttons** (as your LEFT input):

```python
ws.button("X", ws.read_button(8))       # <- YOUR pin numbers
ws.button("Y", ws.read_button(10))
```

*(on the RIGHT: use buttons `"A"` and `"B"`)*

**Tilt sensor** (as your LEFT input — tilting the controller moves the stick):

```python
x, y = ws.read_tilt_percent()
ws.left_stick(x, y)
```

*(on the RIGHT: use `ws.right_stick`)*

In [ ]:
# ================ YOUR GAME LOOP ================
# Swap each half for the snippet that matches YOUR input
# on that side (see the menu above). Keep ws.update() last!

while True:

    # ---- LEFT input  (this example: joystick) ----
    x, y = ws.read_joystick_percent("left")
    ws.left_stick(x, y)
    ws.button("LS", ws.read_button(36))

    # ---- RIGHT input  (this example: two buttons) ----
    ws.button("A", ws.read_button(8))
    ws.button("B", ws.read_button(10))

    ws.update()   # send everything to the game, then repeat

## Step 4 — Play!

While the loop above is running (you'll see its status line updating as you move your inputs), your controller is **live**. Leave it running and:

1. Open **SuperTuxKart** on the Pi.
2. Go to **Options → Controls**. You should see **"Workshop Controller"** in the list — that's you!
3. Click it and bind the actions: when the game asks for *steer left*, *accelerate*, *fire*, and so on, just **move the input you want to use for it**. This is the accessible-gaming payoff — *any* action can go on *any* input, wherever it's easiest for the player.
4. Race!

**To stop the controller:** press the **square stop button** next to the loop cell. (A red `KeyboardInterrupt` message is normal — that's just Python confirming you stopped it.)

### If something's off

- **`NameError: LEFT_INPUT is not defined`** — you skipped pasting your setup cell in Step 1.
- **`/dev/uinput` complaint** — in a terminal run `sudo modprobe uinput`, then re-run the gamepad cell.
- **The game doesn't list "Workshop Controller"** — make sure the loop cell is actually *running* (its status line should be ticking) while you look at the controls menu.
- **An input does nothing in-game** — stop the loop and check your snippet uses the right side (`"left"`/`"right"`) and *your* pin numbers. Lab 1's live test is always there if you want to re-verify the wiring.
- **The kart drifts on its own** — your joystick center or tilt zero is off. For tilt, re-run the connect cell while holding the controller in its resting position.

### Want to experiment?

Try swapping which side does what, or map your encoder's press to a different button name — just edit the loop and run it again. The controller is yours now.